# 음성 합성 탐지 모델링 노트북 — 가독성 개선본

이 노트북은 실제 음성과 합성 음성을 구분하는 이진 분류 프로젝트를 **실행 순서에 맞게 재배치**한 버전입니다.

주요 개선 사항은 다음과 같습니다.

1. 셀 실행 순서를 `환경 설정 → 데이터 확인 → 전처리 → 모델 정의 → 학습 → 결과 분석 → 고도화` 흐름으로 정리했습니다.
2. 각 코드 셀 앞에 “무엇을 하는 셀인지” 설명하는 Markdown 안내 셀을 추가했습니다.
3. 각 코드 셀에는 상세 주석과 `NOTE` 설명을 추가해 복습하기 쉽게 구성했습니다.
4. 결과 해석을 위해 데이터 분포, 음성 길이, Mel 통계, 학습 곡선, 모델 성능 비교, 혼동행렬 등 다양한 그래프 셀을 추가했습니다.
5. 기존 XLS-R + AASIST 확장 코드는 뒤쪽 “고도화 실험” 섹션으로 분리했습니다.


## 목차

1. 프로젝트 개요
2. 환경 설정 및 라이브러리 준비
3. 데이터 경로 설정 및 데이터 목록 생성
4. 데이터 탐색 EDA
5. Mel-Spectrogram 전처리
6. 데이터셋 및 DataLoader 구성
7. 모델 정의: GRU, LCNN, CRNN
8. 모델 학습 및 평가
9. 결과 비교 및 그래프 분석
10. XLS-R + AASIST 고도화 방향
11. 참고 자료


## 1. 프로젝트 개요

실제 음성(`wav_real`)과 합성 음성(`synth_full_*`)을 구분하는 이진 분류 문제로 정의합니다.


### 셀 1: 0. 프로젝트 개요

이 셀은 위 섹션의 실행 흐름에서 필요한 준비 작업을 수행합니다. 주석을 따라 경로와 설정값을 확인한 후 실행하세요.


In [ ]:
# NOTE: 이 셀은 프로젝트 전체 목표와 데이터 구조를 설명하는 안내 셀입니다. 실행 결과보다는 전체 흐름 파악이 목적입니다.
# ============================================================
# 0. 프로젝트 개요
# ============================================================
# 목적:
# - 실제 음성(wav_real)과 합성 음성(synth_full_*)을 구분하는 이진 분류 모델을 학습한다.
#
# 데이터 구조:
# data_kspon/
# ├── wav_real/
# │   ├── train/
# │   ├── val/
# │   └── test/
# ├── synth_full_aug/
# │   ├── train/
# │   ├── val/
# │   └── test/
# ├── synth_full_elevenlabs_aug/
# │   ├── train/
# │   ├── val/
# │   └── test/
# └── synth_full_google_aug/
#     ├── train/
#     ├── val/
#     └── test/
#
# 라벨 정의:
# - wav_real                  → 0 = 실제 음성
# - synth_full_aug            → 1 = 합성 음성
# - synth_full_elevenlabs_aug → 1 = 합성 음성
# - synth_full_google_aug     → 1 = 합성 음성
#
# 본 코드에서 수행하는 작업:
# 1. Google Drive 연결
# 2. 데이터 경로 확인
# 3. 전체 wav 목록 수집
# 4. wav 개수 확인
# 5. 실제/합성 샘플 waveform 확인
# 6. Mel-Spectrogram 생성
# 7. PyTorch Dataset / DataLoader 구성
# 8. GRU 모델 학습
# 9. LCNN 모델 학습
# 10. CRNN 모델 학습
# 11. 성능 비교표 출력
# ============================================================

## 2. 환경 설정 및 라이브러리 준비

Colab/Google Drive/GPU/난수 시드 등 실험 재현성과 실행 환경을 먼저 준비합니다.


### 셀 2: 1. Google Drive 연결

이 셀은 위 섹션의 실행 흐름에서 필요한 준비 작업을 수행합니다. 주석을 따라 경로와 설정값을 확인한 후 실행하세요.


In [ ]:
# NOTE: Colab 환경에서만 실행하세요. 로컬 Jupyter에서는 Google Drive 마운트가 동작하지 않을 수 있습니다.
# ============================================================
# 1. Google Drive 연결
# ============================================================
# Colab에서 Google Drive 안의 데이터셋에 접근하기 위해 Drive를 마운트한다.
# 실행하면 Google 계정 인증 절차가 뜬다.
# 인증이 완료되면 /content/drive/MyDrive 경로로 내 드라이브에 접근 가능하다.

from google.colab import drive
drive.mount('/content/drive')

### 셀 3: 2. 필요한 라이브러리 설치

이 셀은 위 섹션의 실행 흐름에서 필요한 준비 작업을 수행합니다. 주석을 따라 경로와 설정값을 확인한 후 실행하세요.


In [ ]:
# NOTE: 필요한 패키지를 설치합니다. Colab 런타임을 새로 시작하면 다시 실행해야 합니다.
# ============================================================
# 2. 필요한 라이브러리 설치
# ============================================================
# librosa      : wav 음성 파일 로딩 및 Mel-Spectrogram 생성
# soundfile    : 음성 파일 처리 보조 라이브러리
# matplotlib   : waveform, spectrogram, 학습 곡선 시각화
# pandas       : 데이터 목록 및 결과표 관리
# scikit-learn : accuracy, precision, recall, f1-score 등 평가 지표 계산
# tqdm         : 학습 진행률 표시

!pip install librosa soundfile matplotlib pandas scikit-learn tqdm -q

### 셀 4: 3. 라이브러리 import

이 셀은 위 섹션의 실행 흐름에서 필요한 준비 작업을 수행합니다. 주석을 따라 경로와 설정값을 확인한 후 실행하세요.


In [ ]:
# NOTE: 이후 모든 셀에서 사용할 공통 라이브러리를 한 번에 불러옵니다.
# ============================================================
# 3. 라이브러리 import
# ============================================================

import os
import glob
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import librosa
import librosa.display

from tqdm import tqdm
from IPython.display import Audio, display

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

### 셀 5: 4. GPU 사용 여부 확인

이 셀은 위 섹션의 실행 흐름에서 필요한 준비 작업을 수행합니다. 주석을 따라 경로와 설정값을 확인한 후 실행하세요.


In [ ]:
# NOTE: GPU 사용 여부를 확인하여 학습 속도 병목을 사전에 점검합니다.
# ============================================================
# 4. GPU 사용 여부 확인
# ============================================================
# Colab 런타임에서 GPU가 켜져 있으면 cuda가 출력된다.
# 메뉴:
# 런타임 → 런타임 유형 변경 → 하드웨어 가속기 → GPU
#
# 처음 테스트는 T4로 충분하다.
# XLS-R + AASIST처럼 무거운 모델은 A100 권장.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("현재 사용 device:", device)

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))
else:
    print("GPU가 비활성화되어 있습니다. Colab 런타임 설정을 확인하세요.")



### 셀 6: 5. 랜덤 시드 고정

이 셀은 위 섹션의 실행 흐름에서 필요한 준비 작업을 수행합니다. 주석을 따라 경로와 설정값을 확인한 후 실행하세요.


In [ ]:
# NOTE: 재현 가능한 실험을 위해 난수 시드를 고정합니다.
# ============================================================
# 5. 랜덤 시드 고정
# ============================================================
# 모델 학습은 랜덤 초기값, 데이터 셔플 등에 따라 결과가 달라질 수 있다.
# seed를 고정하면 같은 조건에서 최대한 비슷한 결과를 재현할 수 있다.

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # benchmark=True는 입력 크기가 고정되어 있을 때 GPU 연산 속도를 높여준다.
    # 완전한 재현성보다는 학습 속도를 조금 더 우선한다.
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(42)


## 3. 데이터 경로 설정 및 데이터 목록 생성

데이터가 있는 경로를 지정하고, 모든 wav 파일을 탐색하여 `df` DataFrame으로 정리합니다.


### 셀 7: 6. 데이터 경로 설정 및 로컬 복사 (속도 향상)

이 셀은 위 섹션의 실행 흐름에서 필요한 준비 작업을 수행합니다. 주석을 따라 경로와 설정값을 확인한 후 실행하세요.


In [ ]:
# NOTE: Google Drive는 I/O가 느릴 수 있으므로 /content 로컬 경로로 복사해 학습 속도를 개선합니다.
# ============================================================
# 6. 데이터 경로 설정 및 로컬 복사 (속도 향상)
# ============================================================
import os
import shutil

# 원본 구글 드라이브 경로
drive_path = "/content/drive/MyDrive/딥러닝-석흥일/프로젝트_보이스피싱탐지/data_kspon"
# 코랩 로컬 경로
local_path = "/content/data_kspon"

print("구글 드라이브 원본 경로:", drive_path)

# 데이터 로컬 복사 로직 (I/O 병목 해결)
if not os.path.exists(local_path):
    print("\n학습 속도 향상을 위해 구글 드라이브의 데이터를 /content/ 로컬로 복사합니다...")
    print("파일 개수가 많아 수 분 정도 소요될 수 있습니다. 잠시만 기다려주세요.")
    shutil.copytree(drive_path, local_path)
    print("✅ 데이터 로컬 복사 완료!")
else:
    print("\n✅ 이미 로컬에 데이터가 복사되어 있습니다.")

# 학습에 사용할 실제 데이터 경로를 로컬 폴더로 지정
data_path = local_path

print("\n최종 데이터 경로:", data_path)
print("데이터 경로 존재 여부:", os.path.exists(data_path))

if os.path.exists(data_path):
    print("\n최상위 파일/폴더 목록:")
    print(os.listdir(data_path))
else:
    raise FileNotFoundError("data_path가 존재하지 않습니다.")


### 셀 8: 7. 클래스 폴더와 라벨 정의

이 셀은 위 섹션의 실행 흐름에서 필요한 준비 작업을 수행합니다. 주석을 따라 경로와 설정값을 확인한 후 실행하세요.


In [ ]:
# NOTE: 폴더명과 라벨을 명확히 매핑합니다. 실제 음성은 0, 합성 음성은 1입니다.
# ============================================================
# 7. 클래스 폴더와 라벨 정의
# ============================================================
# 이 프로젝트는 이진 분류 문제로 구성한다.
#
# label 0:
# - 실제 사람이 말한 음성
#
# label 1:
# - 네이버 클로바 보이스 합성 음성
# - ElevenLabs 합성 음성
# - Google TTS 합성 음성

class_folders = {
    "wav_real": 0,
    "synth_full_aug": 1,
    "synth_full_elevenlabs_aug": 1,
    "synth_full_google_aug": 1
}

# 이미 데이터가 train / val / test로 나뉘어 있으므로
# 별도의 train_test_split은 수행하지 않는다.
splits = ["train", "val", "test"]


### 셀 9: 8. 전체 wav 파일 목록 수집

이 셀은 위 섹션의 실행 흐름에서 필요한 준비 작업을 수행합니다. 주석을 따라 경로와 설정값을 확인한 후 실행하세요.


In [ ]:
# NOTE: 전체 wav 파일을 탐색하여 모델 학습에 사용할 기본 DataFrame을 만듭니다.
# ============================================================
# 8. 전체 wav 파일 목록 수집
# ============================================================
# 각 클래스 폴더 아래 train / val / test 폴더를 돌면서
# 모든 wav 파일 경로와 라벨을 DataFrame으로 저장한다.

data_list = []

for class_name, label in class_folders.items():
    for split in splits:
        split_path = os.path.join(data_path, class_name, split)

        if not os.path.exists(split_path):
            print("경로 없음:", split_path)
            continue

        # recursive=True와 **/*.wav 패턴을 사용하여 하위 디렉토리의 모든 wav 파일을 수집합니다.
        wav_files = glob.glob(os.path.join(split_path, "**", "*.wav"), recursive=True)

        print(f"{class_name} / {split}: {len(wav_files)}개")

        for wav_path in wav_files:
            data_list.append({
                "path": wav_path,
                "class_name": class_name,
                "split": split,
                "label": label
            })

df = pd.DataFrame(data_list)

if not df.empty:
    print("\n전체 wav 개수:", len(df))
    display(df.head())
else:
    print("\n데이터를 찾지 못했습니다. 경로 설정을 다시 확인해주세요.")

### 셀 10: 9. 데이터 개수 확인

이 셀은 위 섹션의 실행 흐름에서 필요한 준비 작업을 수행합니다. 주석을 따라 경로와 설정값을 확인한 후 실행하세요.


In [ ]:
# NOTE: 클래스별 데이터 수를 확인해 데이터 누락이나 클래스 불균형을 점검합니다.
# ============================================================
# 9. 데이터 개수 확인
# ============================================================
# 클래스별, split별 데이터 개수를 확인한다.
# 이 단계에서 특정 폴더가 0개로 나오면 경로 또는 파일 확장자를 확인해야 한다.

if not df.empty:
    count_table = (
        df.groupby(["class_name", "split", "label"])
          .size()
          .reset_index(name="count")
    )

    print("\n클래스별 / split별 wav 개수:")
    display(count_table)

    print("전체 데이터:", len(df))
    print("실제 음성 개수:", len(df[df["label"] == 0]))
    print("합성 음성 개수:", len(df[df["label"] == 1]))
else:
    print("오류: 데이터프레임이 비어 있습니다. 8번 셀의 파일 경로와 검색 패턴(recursive=True 등)을 확인해 주세요.")

### 셀 11: 10. train / val / test별 개수 확인

이 셀은 위 섹션의 실행 흐름에서 필요한 준비 작업을 수행합니다. 주석을 따라 경로와 설정값을 확인한 후 실행하세요.


In [ ]:
# NOTE: train/val/test 분포를 확인하여 평가 데이터가 충분한지 확인합니다.
# ============================================================
# 10. train / val / test별 개수 확인
# ============================================================
# 모델 학습에는 train을 사용하고,
# 학습 중 성능 확인에는 val을 사용하고,
# 최종 성능 평가는 test를 사용한다.

split_count = (
    df.groupby(["split", "label"])
      .size()
      .reset_index(name="count")
)

print("\ntrain / val / test별 실제/합성 개수:")
display(split_count)

## 4. 데이터 탐색 EDA

이 섹션은 모델을 학습하기 전에 데이터가 정상적으로 구성되어 있는지 확인하는 단계입니다.  
클래스 불균형, split별 데이터 수, 음성 길이 분포를 먼저 확인해야 학습 결과를 더 정확하게 해석할 수 있습니다.


In [ ]:
# ============================================================
# 추가 EDA 1. 클래스 / Split 분포 시각화
# ============================================================
# 목적:
# - 실제 음성과 합성 음성의 데이터 개수가 균형적인지 확인한다.
# - train/val/test 분리가 의도한 대로 되어 있는지 확인한다.
# - 클래스 불균형이 심하면 accuracy만으로 모델을 평가하기 어렵기 때문에
#   precision, recall, f1-score를 함께 봐야 한다.

if not df.empty:
    # label 값을 사람이 이해하기 쉬운 이름으로 변환한다.
    df_plot = df.copy()
    df_plot["label_name"] = df_plot["label"].map({0: "real", 1: "synthetic"})

    # 1) 전체 label 분포
    label_counts = df_plot["label_name"].value_counts().sort_index()
    plt.figure(figsize=(6, 4))
    label_counts.plot(kind="bar")
    plt.title("Overall Label Distribution")
    plt.xlabel("Label")
    plt.ylabel("Number of wav files")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

    # 2) split별 label 분포
    split_label_counts = (
        df_plot.groupby(["split", "label_name"])
               .size()
               .unstack(fill_value=0)
    )
    split_label_counts.plot(kind="bar", figsize=(8, 4))
    plt.title("Label Distribution by Split")
    plt.xlabel("Split")
    plt.ylabel("Number of wav files")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

    # 3) 합성 음성 소스별 분포
    class_counts = df_plot["class_name"].value_counts().sort_index()
    plt.figure(figsize=(9, 4))
    class_counts.plot(kind="bar")
    plt.title("File Count by Source Folder")
    plt.xlabel("Source folder")
    plt.ylabel("Number of wav files")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("df가 비어 있습니다. 데이터 경로와 wav 파일 존재 여부를 먼저 확인하세요.")


In [ ]:
# ============================================================
# 추가 EDA 2. 음성 길이 분포 확인
# ============================================================
# 목적:
# - 실제 음성과 합성 음성의 길이 분포가 크게 다른지 확인한다.
# - 만약 특정 클래스만 길이가 길거나 짧다면 모델이 음성 내용이 아니라 길이 차이를 학습할 수 있다.
# - MAX_LENGTH 설정이 너무 짧으면 중요한 뒷부분이 잘릴 수 있고,
#   너무 길면 padding이 많아져 학습 효율이 떨어질 수 있다.

DURATION_SAMPLE_LIMIT = 1000  # 전체 파일이 많을 때 EDA 시간을 줄이기 위한 샘플 개수 제한

if not df.empty:
    duration_df = df.sample(min(len(df), DURATION_SAMPLE_LIMIT), random_state=42).copy()
    durations = []

    for wav_path in tqdm(duration_df["path"], desc="Calculate duration"):
        try:
            # librosa.get_duration은 전체 waveform을 학습용으로 변환하지 않고 길이만 확인할 때 유용하다.
            duration = librosa.get_duration(path=wav_path)
        except Exception as e:
            # 일부 파일 손상 또는 경로 오류가 있어도 전체 노트북이 중단되지 않도록 NaN 처리한다.
            duration = np.nan
        durations.append(duration)

    duration_df["duration_sec"] = durations
    duration_df["label_name"] = duration_df["label"].map({0: "real", 1: "synthetic"})

    display(duration_df.groupby("label_name")["duration_sec"].describe())

    plt.figure(figsize=(8, 4))
    for label_name in ["real", "synthetic"]:
        subset = duration_df[duration_df["label_name"] == label_name]["duration_sec"].dropna()
        plt.hist(subset, bins=30, alpha=0.5, label=label_name)
    plt.title("Audio Duration Distribution")
    plt.xlabel("Duration seconds")
    plt.ylabel("Number of samples")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("df가 비어 있어 음성 길이 분포를 계산할 수 없습니다.")


## 5. Waveform 및 Mel-Spectrogram 전처리

모델은 wav 원본을 직접 쓰지 않고 Mel-Spectrogram을 입력으로 사용합니다.  
따라서 실제/합성 샘플의 waveform과 Mel-Spectrogram을 눈으로 확인해 데이터가 정상인지 검증합니다.


### 셀 12: 전처리 관련 코드

이 셀은 음성 샘플 확인, Mel-Spectrogram 생성, 데이터 분리 또는 캐싱을 담당합니다. 실행 후 shape와 그래프가 정상적으로 출력되는지 확인하세요.


In [ ]:
# NOTE: 실제 음성 샘플이 정상적으로 로드되는지 waveform으로 검증합니다.
# ============================================================
# 11. 실제 음성 샘플 waveform 확인
# ============================================================
# librosa.load()로 wav 파일을 읽는다.
# sr=16000은 모든 음성을 16kHz로 통일한다는 의미다.
#
# waveform은 시간에 따른 진폭 변화를 보여준다.
# 여기서는 데이터가 정상적으로 읽히는지 확인하는 목적이다.

real_sample = df[
    (df["class_name"] == "wav_real") &
    (df["split"] == "train")
].iloc[0]

real_path = real_sample["path"]

real_audio, real_sr = librosa.load(real_path, sr=16000)

print("\n실제 음성 샘플 경로:", real_path)
print("샘플레이트:", real_sr)
print("오디오 배열 길이:", len(real_audio))
print("재생 시간 초:", len(real_audio) / real_sr)
print("라벨:", real_sample["label"])

plt.figure(figsize=(15, 4))
plt.plot(real_audio)
plt.title("Waveform - Real Voice")
plt.xlabel("Sample")
plt.ylabel("Amplitude")
plt.show()

display(Audio(real_audio, rate=real_sr))



### 셀 13: 전처리 관련 코드

이 셀은 음성 샘플 확인, Mel-Spectrogram 생성, 데이터 분리 또는 캐싱을 담당합니다. 실행 후 shape와 그래프가 정상적으로 출력되는지 확인하세요.


In [ ]:
# NOTE: 합성 음성 샘플도 동일하게 로드하여 실제 음성과 비교할 준비를 합니다.
# ============================================================
# 12. 합성 음성 샘플 waveform 확인
# ============================================================

synth_sample = df[
    (df["label"] == 1) &
    (df["split"] == "train")
].iloc[0]

synth_path = synth_sample["path"]

synth_audio, synth_sr = librosa.load(synth_path, sr=16000)

print("\n합성 음성 샘플 경로:", synth_path)
print("합성 음성 종류:", synth_sample["class_name"])
print("샘플레이트:", synth_sr)
print("오디오 배열 길이:", len(synth_audio))
print("재생 시간 초:", len(synth_audio) / synth_sr)
print("라벨:", synth_sample["label"])

plt.figure(figsize=(15, 4))
plt.plot(synth_audio)
plt.title(f"Waveform - {synth_sample['class_name']}")
plt.xlabel("Sample")
plt.ylabel("Amplitude")
plt.show()

display(Audio(synth_audio, rate=synth_sr))

### 셀 14: 전처리 관련 코드

이 셀은 음성 샘플 확인, Mel-Spectrogram 생성, 데이터 분리 또는 캐싱을 담당합니다. 실행 후 shape와 그래프가 정상적으로 출력되는지 확인하세요.


In [ ]:
# NOTE: Mel-Spectrogram 변환에 사용할 핵심 하이퍼파라미터를 한 곳에서 관리합니다.
# ============================================================
# 13. Mel-Spectrogram 설정값
# ============================================================
# GRU / LCNN / CRNN은 wav 원본을 바로 넣지 않고
# Mel-Spectrogram으로 변환한 뒤 학습한다.
#
# SAMPLE_RATE = 16000:
# - 모든 음성을 16kHz로 통일
#
# N_MELS = 80:
# - 주파수축을 80개의 Mel filter로 요약
#
# N_FFT = 1024:
# - 한 번에 FFT를 수행할 window 크기
#
# HOP_LENGTH = 256:
# - 다음 frame으로 이동하는 간격
#
# MAX_LENGTH = 400:
# - 시간축 길이를 400 frame으로 고정
# - 짧은 음성은 0 padding
# - 긴 음성은 앞부분 400 frame만 사용
#
# 최종 Mel shape:
# - (80, 400)

SAMPLE_RATE = 16000
N_MELS = 80
N_FFT = 1024
HOP_LENGTH = 256
MAX_LENGTH = 400


### 셀 15: 전처리 관련 코드

이 셀은 음성 샘플 확인, Mel-Spectrogram 생성, 데이터 분리 또는 캐싱을 담당합니다. 실행 후 shape와 그래프가 정상적으로 출력되는지 확인하세요.


In [ ]:
# NOTE: wav 파일을 모델 입력 형태인 Mel-Spectrogram 배열로 변환하는 핵심 함수입니다.
# ============================================================
# 14. Mel-Spectrogram 생성 함수
# ============================================================
# wav 파일 1개를 입력받아 모델 입력용 Mel-Spectrogram으로 변환한다.
#
# 처리 순서:
# 1. wav 로딩
# 2. Mel-Spectrogram 계산
# 3. dB scale 변환
# 4. 길이 고정
# 5. 정규화
#
# 반환값:
# - numpy array
# - shape: (N_MELS, MAX_LENGTH)
# - dtype: float32

def make_mel_spectrogram(
    wav_path,
    sr=SAMPLE_RATE,
    n_mels=N_MELS,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    max_length=MAX_LENGTH
):
    audio, _ = librosa.load(wav_path, sr=sr)

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels
    )

    mel_db = librosa.power_to_db(mel, ref=np.max)

    if mel_db.shape[1] < max_length:
        pad_width = max_length - mel_db.shape[1]

        mel_db = np.pad(
            mel_db,
            pad_width=((0, 0), (0, pad_width)),
            mode="constant"
        )
    else:
        mel_db = mel_db[:, :max_length]

    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)

    return mel_db.astype(np.float32)


### 셀 16: 전처리 관련 코드

이 셀은 음성 샘플 확인, Mel-Spectrogram 생성, 데이터 분리 또는 캐싱을 담당합니다. 실행 후 shape와 그래프가 정상적으로 출력되는지 확인하세요.


In [ ]:
# NOTE: 실제 음성의 Mel-Spectrogram을 시각화해 입력 특징을 눈으로 확인합니다.
# ============================================================
# 15. 실제 음성 Mel-Spectrogram 시각화
# ============================================================

real_mel = make_mel_spectrogram(real_path)

print("실제 음성 Mel shape:", real_mel.shape)

plt.figure(figsize=(12, 4))
librosa.display.specshow(
    real_mel,
    sr=SAMPLE_RATE,
    hop_length=HOP_LENGTH,
    x_axis="time",
    y_axis="mel"
)
plt.colorbar()
plt.title("Mel-Spectrogram - Real Voice")
plt.tight_layout()
plt.show()


### 셀 17: 전처리 관련 코드

이 셀은 음성 샘플 확인, Mel-Spectrogram 생성, 데이터 분리 또는 캐싱을 담당합니다. 실행 후 shape와 그래프가 정상적으로 출력되는지 확인하세요.


In [ ]:
# NOTE: 합성 음성의 Mel-Spectrogram을 시각화해 실제 음성과의 패턴 차이를 확인합니다.
# ============================================================
# 16. 합성 음성 Mel-Spectrogram 시각화
# ============================================================

synth_mel = make_mel_spectrogram(synth_path)

print("합성 음성 Mel shape:", synth_mel.shape)

plt.figure(figsize=(12, 4))
librosa.display.specshow(
    synth_mel,
    sr=SAMPLE_RATE,
    hop_length=HOP_LENGTH,
    x_axis="time",
    y_axis="mel"
)
plt.colorbar()
plt.title("Mel-Spectrogram - Synthetic Voice")
plt.tight_layout()
plt.show()



### 셀 18: 전처리 관련 코드

이 셀은 음성 샘플 확인, Mel-Spectrogram 생성, 데이터 분리 또는 캐싱을 담당합니다. 실행 후 shape와 그래프가 정상적으로 출력되는지 확인하세요.


In [ ]:
# NOTE: split 컬럼을 기준으로 학습/검증/테스트 DataFrame을 분리합니다.
# ============================================================
# 17. train / val / test DataFrame 분리
# ============================================================

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

print("train 개수:", len(train_df))
print("val 개수:", len(val_df))
print("test 개수:", len(test_df))

### 셀 19: 전처리 관련 코드

이 셀은 음성 샘플 확인, Mel-Spectrogram 생성, 데이터 분리 또는 캐싱을 담당합니다. 실행 후 shape와 그래프가 정상적으로 출력되는지 확인하세요.


In [ ]:
# NOTE: 빠른 디버깅을 위해 소량 데이터만 사용하는 옵션입니다. 최종 학습은 False 권장입니다.
# ============================================================
# 18. 빠른 테스트용 데이터 축소 옵션
# ============================================================
# 처음부터 전체 데이터로 돌리면 시간이 오래 걸릴 수 있다.
# 따라서 처음에는 USE_SMALL_DATA = True로 두고,
# 각 label별 일부 샘플만 사용해 코드가 정상 동작하는지 확인한다.
#
# 코드가 정상적으로 돌아가면 USE_SMALL_DATA = False로 바꾸고 전체 학습한다.

USE_SMALL_DATA = False

if USE_SMALL_DATA:
    train_df = (
        train_df.groupby("label", group_keys=False)
                .apply(lambda x: x.sample(min(len(x), 200), random_state=42))
                .reset_index(drop=True)
    )

    val_df = (
        val_df.groupby("label", group_keys=False)
              .apply(lambda x: x.sample(min(len(x), 50), random_state=42))
              .reset_index(drop=True)
    )

    test_df = (
        test_df.groupby("label", group_keys=False)
               .apply(lambda x: x.sample(min(len(x), 50), random_state=42))
               .reset_index(drop=True)
    )

print("학습에 사용할 train 개수:", len(train_df))
print("학습에 사용할 val 개수:", len(val_df))
print("학습에 사용할 test 개수:", len(test_df))


### 셀 20: 전처리 관련 코드

이 셀은 음성 샘플 확인, Mel-Spectrogram 생성, 데이터 분리 또는 캐싱을 담당합니다. 실행 후 shape와 그래프가 정상적으로 출력되는지 확인하세요.


In [ ]:
# NOTE: Mel-Spectrogram을 .pt 텐서로 캐싱해 반복 학습 시 전처리 시간을 줄입니다.
import os
import torch
from tqdm import tqdm

# ============================================================
# 18-5. Mel-Spectrogram 사전 계산 및 캐싱 (속도 10배 이상 향상)
# ============================================================
print("============================================================")
print("🚀 1단계: Mel-Spectrogram 사전 계산 및 캐싱 진행")
print("============================================================")
print("전체 오디오 파일을 미리 Mel-Spectrogram 텐서(.pt)로 변환하여 저장합니다.")
print("최초 1회 연산 시간이 소요되지만, 이후 학습 시 CPU 병목이 완전히 사라집니다!\n")

def cache_mels(dataframe):
    for idx, row in tqdm(dataframe.iterrows(), total=len(dataframe)):
        wav_path = row["path"]
        mel_path = wav_path.replace(".wav", ".pt")

        # 이미 변환된 .pt 파일이 존재하지 않을 때만 연산 후 저장
        if not os.path.exists(mel_path):
            mel_data = make_mel_spectrogram(wav_path)
            mel_tensor = torch.tensor(mel_data, dtype=torch.float32)
            torch.save(mel_tensor, mel_path)

print("[Train Data 캐싱]")
cache_mels(train_df)
print("\n[Val Data 캐싱]")
cache_mels(val_df)
print("\n[Test Data 캐싱]")
cache_mels(test_df)
print("\n✅ 모든 데이터 캐싱 완료!\n")

# ============================================================
# 19. PyTorch Dataset 클래스 정의 (캐싱 적용 버전)
# ============================================================
# 기존에는 매번 make_mel_spectrogram()을 호출하여 CPU 연산이 발생했지만,
# 이제는 저장된 .pt 파일만 불러오므로 데이터를 읽는 속도가 비약적으로 상승합니다.

class VoiceMelDataset(Dataset):
    def __init__(self, dataframe, model_type="cnn"):
        self.df = dataframe.reset_index(drop=True)
        self.model_type = model_type

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        wav_path = row["path"]
        label = row["label"]

        # 저장된 .pt 텐서 파일 경로
        mel_path = wav_path.replace(".wav", ".pt")

        # 연산 없이 파일에서 바로 텐서 로드! (GPU를 굶기지 않음)
        mel = torch.load(mel_path)
        label = torch.tensor(label, dtype=torch.long)

        if self.model_type == "rnn":
            mel = mel.transpose(0, 1)
        elif self.model_type == "cnn":
            mel = mel.unsqueeze(0)

        return mel, label


In [ ]:
# ============================================================
# 추가 EDA 3. 실제/합성 Mel-Spectrogram 통계 비교
# ============================================================
# 목적:
# - Mel-Spectrogram 값의 평균/표준편차가 클래스별로 크게 다른지 확인한다.
# - 전처리 정규화가 잘 수행되면 평균은 0 근처, 표준편차는 1 근처로 맞춰진다.
# - 이 그래프는 모델이 입력받는 특징값의 분포를 이해하는 데 도움이 된다.

if "real_mel" in globals() and "synth_mel" in globals():
    mel_stat_df = pd.DataFrame([
        {"sample": "real", "mean": float(np.mean(real_mel)), "std": float(np.std(real_mel)), "min": float(np.min(real_mel)), "max": float(np.max(real_mel))},
        {"sample": "synthetic", "mean": float(np.mean(synth_mel)), "std": float(np.std(synth_mel)), "min": float(np.min(synth_mel)), "max": float(np.max(synth_mel))},
    ])

    display(mel_stat_df)

    for metric in ["mean", "std", "min", "max"]:
        plt.figure(figsize=(5, 3))
        plt.bar(mel_stat_df["sample"], mel_stat_df[metric])
        plt.title(f"Mel-Spectrogram {metric} Comparison")
        plt.xlabel("Sample type")
        plt.ylabel(metric)
        plt.tight_layout()
        plt.show()
else:
    print("real_mel 또는 synth_mel 변수가 없습니다. Mel-Spectrogram 시각화 셀을 먼저 실행하세요.")


## 6. 모델 학습 함수와 모델 정의

이 섹션에서는 모델별로 공통으로 사용하는 학습/평가 함수와 GRU, LCNN, CRNN 모델 구조를 정의합니다.  
정의 셀은 실행해도 바로 학습되지는 않으며, 이후 학습 실행 셀에서 사용됩니다.


### 셀 21: 모델 학습 준비

이 셀은 학습 함수, 평가 함수, 모델 클래스 또는 DataLoader를 정의합니다. 이후 학습 실행 셀들이 이 정의를 사용합니다.


In [ ]:
# NOTE: 모델 1개를 1 epoch 학습시키는 공통 함수입니다.
# ============================================================
# 20. 공통 학습 함수
# ============================================================
# 모델을 1 epoch 학습한다.
#
# 수행 과정:
# 1. batch 단위 데이터 로딩
# 2. GPU로 이동
# 3. 모델 예측
# 4. loss 계산
# 5. 역전파
# 6. optimizer step
# 7. accuracy 계산

def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()

    total_loss = 0
    all_preds = []
    all_labels = []

    for x, y in tqdm(dataloader):
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(x)
        loss = criterion(outputs, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(y.detach().cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)

    return avg_loss, acc


### 셀 22: 모델 학습 준비

이 셀은 학습 함수, 평가 함수, 모델 클래스 또는 DataLoader를 정의합니다. 이후 학습 실행 셀들이 이 정의를 사용합니다.


In [ ]:
# NOTE: 검증/테스트 데이터셋에서 손실과 분류 지표를 계산하는 공통 함수입니다.
# ============================================================
# 21. 공통 평가 함수
# ============================================================
# validation 또는 test 데이터에 대해 모델 성능을 평가한다.
#
# 평가 지표:
# - accuracy  : 전체 중 맞춘 비율
# - precision : 합성이라고 예측한 것 중 실제 합성인 비율
# - recall    : 실제 합성 중 합성이라고 잘 잡은 비율
# - f1        : precision과 recall의 조화평균
# - confusion matrix:
#   [[실제 real을 real로 예측, 실제 real을 synthetic으로 예측],
#    [실제 synthetic을 real로 예측, 실제 synthetic을 synthetic으로 예측]]

def evaluate(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in tqdm(dataloader):
            x = x.to(device)
            y = y.to(device)

            outputs = model(x)
            loss = criterion(outputs, y)

            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(y.detach().cpu().numpy())

    avg_loss = total_loss / len(dataloader)

    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)

    return {
        "loss": avg_loss,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": cm,
        "labels": all_labels,
        "preds": all_preds
    }


### 셀 23: 모델 학습 준비

이 셀은 학습 함수, 평가 함수, 모델 클래스 또는 DataLoader를 정의합니다. 이후 학습 실행 셀들이 이 정의를 사용합니다.


In [ ]:
# NOTE: 학습, 검증, Best 모델 저장, 테스트 평가를 하나로 묶은 실행 함수입니다.
# ============================================================
# 22. 전체 학습 루프 함수
# ============================================================
# 모델 하나에 대해 train → validation → best model 저장 → test 평가까지 수행한다.
#
# best model 기준:
# - validation F1-score가 가장 높은 epoch의 모델을 저장한다.
#
# 저장 파일:
# - 구글 드라이브 경로에 모델 가중치 저장

def run_training(
    model,
    train_loader,
    val_loader,
    test_loader,
    model_name,
    epochs=5,
    lr=1e-4
):
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_f1 = 0
    # 저장 경로 구글 드라이브로 변경
    save_dir = "/content/drive/MyDrive/딥러닝-석흥일/프로젝트_보이스피싱탐지"
    best_model_path = f"{save_dir}/{model_name}_best.pt"

    history = []

    for epoch in range(1, epochs + 1):
        print(f"\n===== {model_name} Epoch {epoch}/{epochs} =====")

        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device
        )

        val_result = evaluate(
            model,
            val_loader,
            criterion,
            device
        )

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(
            f"Val Loss: {val_result['loss']:.4f} | "
            f"Val Acc: {val_result['accuracy']:.4f} | "
            f"Val Precision: {val_result['precision']:.4f} | "
            f"Val Recall: {val_result['recall']:.4f} | "
            f"Val F1: {val_result['f1']:.4f}"
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_result["loss"],
            "val_acc": val_result["accuracy"],
            "val_precision": val_result["precision"],
            "val_recall": val_result["recall"],
            "val_f1": val_result["f1"]
        })

        if val_result["f1"] > best_val_f1:
            best_val_f1 = val_result["f1"]
            torch.save(model.state_dict(), best_model_path)
            print("Best model saved:", best_model_path)

    model.load_state_dict(torch.load(best_model_path, map_location=device))

    test_result = evaluate(
        model,
        test_loader,
        criterion,
        device
    )

    print(f"\n===== {model_name} Test Result =====")
    print("Accuracy :", test_result["accuracy"])
    print("Precision:", test_result["precision"])
    print("Recall   :", test_result["recall"])
    print("F1-score :", test_result["f1"])
    print("Confusion Matrix:")
    print(test_result["confusion_matrix"])

    print("\nClassification Report:")
    print(
        classification_report(
            test_result["labels"],
            test_result["preds"],
            target_names=["real", "synthetic"],
            zero_division=0
        )
    )

    return model, pd.DataFrame(history), test_result


### 셀 24: 모델 학습 준비

이 셀은 학습 함수, 평가 함수, 모델 클래스 또는 DataLoader를 정의합니다. 이후 학습 실행 셀들이 이 정의를 사용합니다.


In [ ]:
# NOTE: GRU 기반 시계열 분류 모델입니다. Mel 주파수 벡터의 시간 흐름을 학습합니다.
# ============================================================
# 23. GRU 모델 정의
# ============================================================
# GRU는 음성의 시간 흐름을 학습하는 RNN 계열 모델이다.
#
# 입력:
# - shape: (batch, time, n_mels)
# - 예: (32, 400, 80)
#
# 구조:
# - Bidirectional GRU
# - 마지막 time step 출력 사용
# - Fully Connected Layer로 실제/합성 분류

class GRUClassifier(nn.Module):
    def __init__(
        self,
        input_size=80,
        hidden_size=128,
        num_layers=2,
        num_classes=2,
        dropout=0.3
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        output, hidden = self.gru(x)

        last_output = output[:, -1, :]

        logits = self.classifier(last_output)

        return logits

### 셀 25: 모델 학습 준비

이 셀은 학습 함수, 평가 함수, 모델 클래스 또는 DataLoader를 정의합니다. 이후 학습 실행 셀들이 이 정의를 사용합니다.


In [ ]:
# NOTE: LCNN 기반 이미지형 분류 모델입니다. Mel-Spectrogram을 2D 이미지처럼 처리합니다.
# ============================================================
# 24. LCNN 모델 정의
# ============================================================
# LCNN은 Light CNN 구조다.
# 음성 Mel-Spectrogram을 이미지처럼 보고 CNN으로 특징을 추출한다.
#
# 핵심:
# - 일반 CNN보다 가볍게 구성
# - MaxFeatureMap을 사용해 채널을 절반으로 줄이면서 중요한 특징만 남긴다.
#
# 입력:
# - shape: (batch, 1, n_mels, time)
# - 예: (32, 1, 80, 400)

class MaxFeatureMap2D(nn.Module):
    def forward(self, x):
        out = torch.chunk(x, 2, dim=1)
        return torch.max(out[0], out[1])


class LCNNClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=1, padding=2),
            MaxFeatureMap2D(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(16, 64, kernel_size=3, stride=1, padding=1),
            MaxFeatureMap2D(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 128, kernel_size=3, stride=1, padding=1),
            MaxFeatureMap2D(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            MaxFeatureMap2D(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        logits = self.classifier(x)
        return logits

### 셀 26: 모델 학습 준비

이 셀은 학습 함수, 평가 함수, 모델 클래스 또는 DataLoader를 정의합니다. 이후 학습 실행 셀들이 이 정의를 사용합니다.


In [ ]:
# NOTE: CRNN은 CNN으로 특징을 뽑고 GRU로 시간 흐름을 학습하는 결합 모델입니다.
# ============================================================
# 25. CRNN 모델 정의
# ============================================================
# CRNN은 CNN + RNN 결합 모델이다.
#
# CNN 역할:
# - Mel-Spectrogram에서 지역적 음성 특징 추출
#
# GRU 역할:
# - CNN이 추출한 특징의 시간 흐름 학습
#
# 입력:
# - shape: (batch, 1, 80, 400)
#
# CNN 통과 후:
# - frequency 축이 줄어들고 channel이 증가한다.
# - 이를 time sequence 형태로 바꿔 GRU에 넣는다.

class CRNNClassifier(nn.Module):
    def __init__(
        self,
        num_classes=2,
        gru_hidden=128,
        dropout=0.3
    ):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2)),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2)),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2))
        )

        self.gru = nn.GRU(
            input_size=128 * 10,
            hidden_size=gru_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.classifier = nn.Sequential(
            nn.Linear(gru_hidden * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.cnn(x)

        b, c, f, t = x.size()

        x = x.permute(0, 3, 1, 2)
        x = x.contiguous().view(b, t, c * f)

        output, hidden = self.gru(x)

        last_output = output[:, -1, :]

        logits = self.classifier(last_output)

        return logits

### 셀 27: 모델 학습 준비

이 셀은 학습 함수, 평가 함수, 모델 클래스 또는 DataLoader를 정의합니다. 이후 학습 실행 셀들이 이 정의를 사용합니다.


In [ ]:
# NOTE: 모델 유형별 입력 차원에 맞춰 Dataset과 DataLoader를 생성합니다.
# ============================================================
# 26. DataLoader 생성
# ============================================================
# batch_size:
# - 한 번에 GPU로 넣는 데이터 개수
# - A100 80GB 활용을 위해 기존 32에서 512로 대폭 상향
#
# num_workers:
# - 데이터를 불러오는 프로세스 개수
# - Colab A100 환경에 맞게 4로 상향

BATCH_SIZE = 512

train_dataset_gru = VoiceMelDataset(train_df, model_type="rnn")
val_dataset_gru = VoiceMelDataset(val_df, model_type="rnn")
test_dataset_gru = VoiceMelDataset(test_df, model_type="rnn")

train_loader_gru = DataLoader(
    train_dataset_gru,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4
)

val_loader_gru = DataLoader(
    val_dataset_gru,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4
)

test_loader_gru = DataLoader(
    test_dataset_gru,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4
)

train_dataset_cnn = VoiceMelDataset(train_df, model_type="cnn")
val_dataset_cnn = VoiceMelDataset(val_df, model_type="cnn")
test_dataset_cnn = VoiceMelDataset(test_df, model_type="cnn")

train_loader_cnn = DataLoader(
    train_dataset_cnn,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4
)

val_loader_cnn = DataLoader(
    val_dataset_cnn,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4
)

test_loader_cnn = DataLoader(
    test_dataset_cnn,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4
)

## 7. 모델 학습 실행

GRU → LCNN → CRNN 순서로 학습합니다.  
처음 실행할 때는 `USE_SMALL_DATA = True`, `epochs=1~2`로 전체 파이프라인을 확인한 뒤 전체 학습을 권장합니다.


### 셀 28: 모델 학습 실행

이 셀은 실제로 GPU/CPU 연산이 오래 걸릴 수 있는 학습 셀입니다. 학습 로그에서 train loss, validation F1 변화를 확인하세요.


In [ ]:
# NOTE: GRU 모델 학습 실행 셀입니다. 먼저 이 모델부터 실행해 전체 파이프라인을 확인하세요.
# ============================================================
# 27. GRU 학습
# ============================================================

gru_model = GRUClassifier()

gru_model, gru_history, gru_test_result = run_training(
    model=gru_model,
    train_loader=train_loader_gru,
    val_loader=val_loader_gru,
    test_loader=test_loader_gru,
    model_name="GRU",
    epochs=20, # 전체 학습을 위해 에폭 증가
    lr=1e-4
)


### 셀 29: 모델 학습 실행

이 셀은 실제로 GPU/CPU 연산이 오래 걸릴 수 있는 학습 셀입니다. 학습 로그에서 train loss, validation F1 변화를 확인하세요.


In [ ]:
# NOTE: LCNN 모델 학습 실행 셀입니다. CNN 입력용 DataLoader를 사용합니다.
# ============================================================
# 28. LCNN 학습
# ============================================================

lcnn_model = LCNNClassifier()

lcnn_model, lcnn_history, lcnn_test_result = run_training(
    model=lcnn_model,
    train_loader=train_loader_cnn,
    val_loader=val_loader_cnn,
    test_loader=test_loader_cnn,
    model_name="LCNN",
    epochs=20, # 전체 학습을 위해 에폭 증가
    lr=1e-4
)


### 셀 30: 모델 학습 실행

이 셀은 실제로 GPU/CPU 연산이 오래 걸릴 수 있는 학습 셀입니다. 학습 로그에서 train loss, validation F1 변화를 확인하세요.


In [ ]:
# NOTE: CRNN 모델 학습 실행 셀입니다. CNN+RNN 결합 구조의 성능을 확인합니다.
# ============================================================
# 29. CRNN 학습
# ============================================================

crnn_model = CRNNClassifier()

crnn_model, crnn_history, crnn_test_result = run_training(
    model=crnn_model,
    train_loader=train_loader_cnn,
    val_loader=val_loader_cnn,
    test_loader=test_loader_cnn,
    model_name="CRNN",
    epochs=20, # 전체 학습을 위해 에폭 증가
    lr=1e-4
)


## 8. 결과 비교 및 그래프 분석

세 모델의 최종 테스트 성능과 학습 과정을 다양한 그래프로 비교합니다.  
보이스피싱/합성음성 탐지에서는 단순 accuracy보다 **recall**과 **F1-score**를 함께 보는 것이 중요합니다.


### 셀 31: 결과 정리 및 기본 시각화

이 셀은 학습된 모델의 성능을 표와 그래프로 확인하는 단계입니다.


In [ ]:
# NOTE: 세 모델의 테스트 성능을 하나의 표로 모읍니다.
# ============================================================
# 30. 모델 성능 비교표 생성
# ============================================================

result_summary = pd.DataFrame([
    {
        "model": "GRU",
        "accuracy": gru_test_result["accuracy"],
        "precision": gru_test_result["precision"],
        "recall": gru_test_result["recall"],
        "f1": gru_test_result["f1"]
    },
    {
        "model": "LCNN",
        "accuracy": lcnn_test_result["accuracy"],
        "precision": lcnn_test_result["precision"],
        "recall": lcnn_test_result["recall"],
        "f1": lcnn_test_result["f1"]
    },
    {
        "model": "CRNN",
        "accuracy": crnn_test_result["accuracy"],
        "precision": crnn_test_result["precision"],
        "recall": crnn_test_result["recall"],
        "f1": crnn_test_result["f1"]
    }
])

print("\n모델별 최종 성능 비교:")
display(result_summary)


### 셀 32: 결과 정리 및 기본 시각화

이 셀은 학습된 모델의 성능을 표와 그래프로 확인하는 단계입니다.


In [ ]:
# NOTE: 최종 성능 비교표를 CSV 파일로 저장합니다.
# ============================================================
# 31. 결과 CSV 저장
# ============================================================

save_path = "/content/drive/MyDrive/딥러닝-석흥일/프로젝트_보이스피싱탐지/voice_detection_model_results.csv"

result_summary.to_csv(save_path, index=False, encoding="utf-8-sig")

print("성능 비교표 저장 완료:", save_path)


### 셀 33: 결과 정리 및 기본 시각화

이 셀은 학습된 모델의 성능을 표와 그래프로 확인하는 단계입니다.


In [ ]:
# NOTE: 모델별 학습 곡선을 시각화하는 기본 함수입니다.
# ============================================================
# 32. 학습 곡선 시각화 함수
# ============================================================

def plot_history(history_df, model_name):
    plt.figure(figsize=(10, 4))
    plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
    plt.title(f"{model_name} Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(history_df["epoch"], history_df["train_acc"], label="train_acc")
    plt.plot(history_df["epoch"], history_df["val_acc"], label="val_acc")
    plt.title(f"{model_name} Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(history_df["epoch"], history_df["val_f1"], label="val_f1")
    plt.title(f"{model_name} Validation F1")
    plt.xlabel("Epoch")
    plt.ylabel("F1-score")
    plt.legend()
    plt.show()


plot_history(gru_history, "GRU")
plot_history(lcnn_history, "LCNN")
plot_history(crnn_history, "CRNN")


### 셀 34: 결과 정리 및 기본 시각화

이 셀은 학습된 모델의 성능을 표와 그래프로 확인하는 단계입니다.


In [ ]:
# NOTE: 혼동행렬을 통해 실제/합성 오분류 유형을 확인합니다.
# ============================================================
# 33. Confusion Matrix 시각화 함수
# ============================================================

def show_confusion_matrix(result, model_name):
    cm = result["confusion_matrix"]

    plt.figure(figsize=(5, 4))
    plt.imshow(cm)
    plt.title(f"{model_name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")

    plt.xticks([0, 1], ["real", "synthetic"])
    plt.yticks([0, 1], ["real", "synthetic"])

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j], ha="center", va="center")

    plt.colorbar()
    plt.show()


show_confusion_matrix(gru_test_result, "GRU")
show_confusion_matrix(lcnn_test_result, "LCNN")
show_confusion_matrix(crnn_test_result, "CRNN")

In [ ]:
# ============================================================
# 추가 결과 그래프 1. 모델별 주요 지표 막대그래프
# ============================================================
# 목적:
# - GRU, LCNN, CRNN의 accuracy / precision / recall / f1을 한눈에 비교한다.
# - 합성 음성 탐지에서는 recall이 낮으면 실제 합성 음성을 놓치는 문제가 생긴다.
# - precision이 낮으면 정상 음성을 합성으로 오탐하는 문제가 생긴다.

if "result_summary" in globals():
    metric_cols = ["accuracy", "precision", "recall", "f1"]
    result_summary_indexed = result_summary.set_index("model")[metric_cols]

    result_summary_indexed.plot(kind="bar", figsize=(10, 5))
    plt.title("Test Metrics by Model")
    plt.xlabel("Model")
    plt.ylabel("Score")
    plt.ylim(0, 1.05)
    plt.xticks(rotation=0)
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.show()
else:
    print("result_summary가 없습니다. 모델 성능 비교표 생성 셀을 먼저 실행하세요.")


In [ ]:
# ============================================================
# 추가 결과 그래프 2. 모델별 Validation F1 추이 비교
# ============================================================
# 목적:
# - epoch가 증가할 때 검증 F1이 개선되는지 확인한다.
# - train 성능만 좋아지고 val F1이 정체/하락하면 과적합 가능성이 있다.
# - 가장 안정적으로 상승하는 모델이 실전 적용 후보가 될 가능성이 높다.

history_map = {}
if "gru_history" in globals():
    history_map["GRU"] = gru_history
if "lcnn_history" in globals():
    history_map["LCNN"] = lcnn_history
if "crnn_history" in globals():
    history_map["CRNN"] = crnn_history

if history_map:
    plt.figure(figsize=(10, 4))
    for model_name, history_df in history_map.items():
        plt.plot(history_df["epoch"], history_df["val_f1"], marker="o", label=model_name)
    plt.title("Validation F1 Trend Comparison")
    plt.xlabel("Epoch")
    plt.ylabel("Validation F1")
    plt.ylim(0, 1.05)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("학습 history가 없습니다. 각 모델 학습 셀을 먼저 실행하세요.")


In [ ]:
# ============================================================
# 추가 결과 그래프 3. 모델별 Loss 추이 비교
# ============================================================
# 목적:
# - train_loss와 val_loss를 모델별로 비교한다.
# - train_loss는 계속 감소하지만 val_loss가 증가하면 과적합 신호일 수 있다.
# - val_loss가 안정적으로 낮아지는 모델이 일반화 성능 측면에서 유리하다.

if history_map:
    for loss_col, title in [("train_loss", "Train Loss"), ("val_loss", "Validation Loss")]:
        plt.figure(figsize=(10, 4))
        for model_name, history_df in history_map.items():
            plt.plot(history_df["epoch"], history_df[loss_col], marker="o", label=model_name)
        plt.title(f"{title} Trend Comparison")
        plt.xlabel("Epoch")
        plt.ylabel(title)
        plt.legend()
        plt.tight_layout()
        plt.show()
else:
    print("학습 history가 없습니다. 각 모델 학습 셀을 먼저 실행하세요.")


In [ ]:
# ============================================================
# 추가 결과 그래프 4. 모든 모델 Confusion Matrix 한 번에 출력
# ============================================================
# 목적:
# - 각 모델이 real → synthetic으로 오탐하는지,
#   synthetic → real로 미탐하는지 비교한다.
# - 보이스피싱/합성음성 탐지에서는 synthetic을 real로 놓치는 FN이 특히 중요하다.

result_map = {}
if "gru_test_result" in globals():
    result_map["GRU"] = gru_test_result
if "lcnn_test_result" in globals():
    result_map["LCNN"] = lcnn_test_result
if "crnn_test_result" in globals():
    result_map["CRNN"] = crnn_test_result

if result_map:
    for model_name, result in result_map.items():
        cm = result["confusion_matrix"]
        plt.figure(figsize=(5, 4))
        plt.imshow(cm)
        plt.title(f"{model_name} Confusion Matrix")
        plt.xlabel("Predicted label")
        plt.ylabel("True label")
        plt.xticks([0, 1], ["real", "synthetic"])
        plt.yticks([0, 1], ["real", "synthetic"])

        for row_idx in range(cm.shape[0]):
            for col_idx in range(cm.shape[1]):
                plt.text(col_idx, row_idx, cm[row_idx, col_idx], ha="center", va="center")

        plt.colorbar()
        plt.tight_layout()
        plt.show()
else:
    print("테스트 결과가 없습니다. 모델 학습 및 평가 셀을 먼저 실행하세요.")


In [ ]:
# ============================================================
# 추가 결과 그래프 5. 모델별 최고 Validation F1 요약
# ============================================================
# 목적:
# - 테스트 결과뿐 아니라 학습 과정 중 가장 좋았던 validation F1도 비교한다.
# - validation F1이 높은데 test F1이 낮다면 데이터 분포 차이 또는 과적합 가능성을 의심할 수 있다.

if history_map:
    best_val_rows = []
    for model_name, history_df in history_map.items():
        best_idx = history_df["val_f1"].idxmax()
        best_row = history_df.loc[best_idx]
        best_val_rows.append({
            "model": model_name,
            "best_epoch": int(best_row["epoch"]),
            "best_val_f1": float(best_row["val_f1"]),
            "best_val_acc": float(best_row["val_acc"]),
        })

    best_val_df = pd.DataFrame(best_val_rows)
    display(best_val_df)

    plt.figure(figsize=(7, 4))
    plt.bar(best_val_df["model"], best_val_df["best_val_f1"])
    plt.title("Best Validation F1 by Model")
    plt.xlabel("Model")
    plt.ylabel("Best Validation F1")
    plt.ylim(0, 1.05)
    plt.tight_layout()
    plt.show()
else:
    print("학습 history가 없습니다. 각 모델 학습 셀을 먼저 실행하세요.")


In [ ]:
# ============================================================
# 추가 결과 그래프 6. 결과 해석 자동 코멘트
# ============================================================
# 목적:
# - result_summary를 기준으로 최고 성능 모델을 자동으로 찾아준다.
# - 보고서 작성 시 어떤 모델을 선택해야 하는지 근거를 만들기 쉽다.

if "result_summary" in globals():
    best_f1_row = result_summary.loc[result_summary["f1"].idxmax()]
    best_recall_row = result_summary.loc[result_summary["recall"].idxmax()]

    print("[최종 결과 해석]")
    print(f"- F1-score 기준 최고 모델: {best_f1_row['model']} / F1 = {best_f1_row['f1']:.4f}")
    print(f"- Recall 기준 최고 모델: {best_recall_row['model']} / Recall = {best_recall_row['recall']:.4f}")
    print("- 실제 운영에서 합성 음성을 놓치는 비용이 크다면 recall을 더 중요하게 볼 수 있습니다.")
    print("- 정상 음성을 합성으로 오탐하는 비용이 크다면 precision도 함께 고려해야 합니다.")
else:
    print("result_summary가 없습니다. 모델 성능 비교표 생성 셀을 먼저 실행하세요.")


## 9. XLS-R + AASIST 고도화 실험

GRU/LCNN/CRNN은 Mel-Spectrogram 기반 모델입니다.  
반면 XLS-R + AASIST는 raw waveform을 사전학습 음성 모델에 넣어 embedding을 추출하고, 그 embedding을 spoofing 탐지 모델에 연결하는 방식입니다.

주의: 이 섹션은 GPU 메모리 사용량이 크므로, 기본 모델 실험이 끝난 뒤 별도로 실행하는 것을 권장합니다.


### 셀 35: XLS-R + AASIST 고도화 관련

이 셀은 사전학습 음성 모델을 활용한 고도화 실험 단계입니다. 기본 모델보다 무겁기 때문에 별도 실행을 권장합니다.


In [ ]:
# NOTE: 고도화 모델인 XLS-R + AASIST로 확장하는 방향을 설명합니다.
# ============================================================
# 34. 참고: XLS-R + AASIST 진행 방향
# ============================================================
# 위 코드는 GRU / LCNN / CRNN 모델을 Mel-Spectrogram 기반으로 구현한 것이다.
#
# XLS-R + AASIST는 구조가 다르다.
#
# GRU / LCNN / CRNN:
# - wav → Mel-Spectrogram → 모델 입력
#
# XLS-R + AASIST:
# - wav → raw waveform
# - XLS-R 사전학습 모델로 음성 embedding 추출
# - AASIST 구조로 spoofing / synthetic detection 수행
#
# 따라서 XLS-R + AASIST는 별도 코드로 구현하는 것이 좋다.
# 먼저 위 3개 모델을 완성한 뒤, 고도화 모델로 XLS-R + AASIST를 붙이는 순서를 추천한다.
# ============================================================

### 셀 36: XLS-R + AASIST 고도화 관련

이 셀은 사전학습 음성 모델을 활용한 고도화 실험 단계입니다. 기본 모델보다 무겁기 때문에 별도 실행을 권장합니다.


In [ ]:
# NOTE: XLS-R 기반 고도화 실험에 필요한 라이브러리와 모델 설정입니다.
# ============================================================
# 35. XLS-R + AASIST 개요
# ============================================================
# XLS-R + AASIST는 최근 음성 위변조 / spoofing 탐지 분야에서 매우 강력한 성능을 보이는 구조입니다.
# 기존 방식과 달리 사전학습 모델의 음성 embedding을 추출하여 사용합니다.

# ============================================================
# 36. XLS-R 관련 라이브러리 설치
# ============================================================
!pip install transformers torchaudio sentencepiece -q

# ============================================================
# 37. XLS-R 관련 라이브러리 import
# ============================================================
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model
import torchaudio

# ============================================================
# 38. XLS-R Feature Extractor 로드
# ============================================================
XLSR_MODEL_NAME = "facebook/wav2vec2-xls-r-300m"

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(XLSR_MODEL_NAME)
# output_attentions=True와 attn_implementation='eager'를 추가하여 어텐션 가중치를 얻도록 설정합니다.
xlsr_model = Wav2Vec2Model.from_pretrained(XLSR_MODEL_NAME, output_attentions=True, attn_implementation='eager')
xlsr_model = xlsr_model.to(device)

print("XLS-R 모델 로딩 완료")

### 셀 42: XLS-R + AASIST 고도화 관련

이 셀은 사전학습 음성 모델을 활용한 고도화 실험 단계입니다. 기본 모델보다 무겁기 때문에 별도 실행을 권장합니다.


In [ ]:
# NOTE: raw waveform을 입력으로 쓰는 XLS-R 전용 Dataset/DataLoader입니다.
import torchaudio
import torch.nn.functional as F

# XLS-R 모델에 필요한 상수 정의 (이미 위에 정의된 경우 재정의는 불필요하지만, 명시적으로)
XLSR_SAMPLE_RATE = 16000 # XLS-R 모델은 16kHz 샘플레이트를 사용
MAX_AUDIO_SECONDS = 5 # 최대 오디오 길이 (초)
MAX_AUDIO_LENGTH = XLSR_SAMPLE_RATE * MAX_AUDIO_SECONDS # 최대 오디오 샘플 수
XLSR_BATCH_SIZE = 4 # XLS-R은 리소스 사용량이 높아 작은 배치 사이즈 사용

class VoiceXLSRDataset(Dataset):
    def __init__(self, dataframe, sampling_rate=XLSR_SAMPLE_RATE, max_length=MAX_AUDIO_LENGTH):
        self.df = dataframe.reset_index(drop=True)
        self.sampling_rate = sampling_rate
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        wav_path = row["path"]
        label = row["label"]

        # torchaudio를 사용하여 오디오 로드 (스테레오 채널이 있을 수 있으므로 첫 번째 채널만 사용)
        audio, sr = torchaudio.load(wav_path)

        # 16kHz로 리샘플링
        if sr != self.sampling_rate:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=self.sampling_rate)
            audio = resampler(audio)

        # 모노 채널로 변환 (만약 여러 채널인 경우)
        if audio.shape[0] > 1:
            audio = audio.mean(dim=0, keepdim=True)

        # 최대 길이로 자르거나 패딩
        if audio.shape[1] > self.max_length:
            audio = audio[:, :self.max_length]
        else:
            padding = self.max_length - audio.shape[1]
            audio = F.pad(audio, (0, padding), "constant", 0)

        # Wav2Vec2FeatureExtractor는 1D numpy array를 기대하므로 squeeze(0) 및 .numpy() 적용
        # 그러나 모델에 직접 넣을 때는 torch.tensor로 다시 변환해야 함.
        # Feature Extractor를 거치는 부분은 모델 학습 루프 내에서 처리하는 것이 일반적.
        # Dataset에서는 raw waveform을 반환.
        audio = audio.squeeze(0) # (1, sequence_length) -> (sequence_length)

        return audio, torch.tensor(label, dtype=torch.long)


print("XLS-R 전용 데이터셋 생성")
train_dataset_xlsr = VoiceXLSRDataset(train_df)
val_dataset_xlsr = VoiceXLSRDataset(val_df)
test_dataset_xlsr = VoiceXLSRDataset(test_df)

train_loader_xlsr = DataLoader(
    train_dataset_xlsr,
    batch_size=XLSR_BATCH_SIZE,
    shuffle=True,
    num_workers=4
)

val_loader_xlsr = DataLoader(
    val_dataset_xlsr,
    batch_size=XLSR_BATCH_SIZE,
    shuffle=False,
    num_workers=4
)

test_loader_xlsr = DataLoader(
    test_dataset_xlsr,
    batch_size=XLSR_BATCH_SIZE,
    shuffle=False,
    num_workers=4
)

print(f"XLS-R Train DataLoader 준비 완료 (Batch Size: {XLSR_BATCH_SIZE})")

### 셀 44: XLS-R + AASIST 고도화 관련

이 셀은 사전학습 음성 모델을 활용한 고도화 실험 단계입니다. 기본 모델보다 무겁기 때문에 별도 실행을 권장합니다.


In [ ]:
# NOTE: XLS-R 임베딩을 받아 분류하는 AASIST 변형 모델입니다.
import torch
import torch.nn as nn
import torch.nn.functional as F

# AASIST 모델 구조는 XLS-R의 임베딩을 입력으로 받아 스푸핑 탐지를 수행합니다.
# AASIST는 원래 Mel-Spectrogram을 입력으로 받지만, 여기서는 XLS-R 임베딩에 맞게 수정합니다.
# XLS-R의 hidden_size는 1024입니다. (facebook/wav2vec2-xls-r-300m 기준)

class AASISTEmbeddingsClassifier(nn.Module):
    def __init__(
        self,
        xlsr_model_dimension=1024, # XLS-R 모델의 출력 임베딩 차원
        num_classes=2,
        dropout=0.3
    ):
        super().__init__()

        # Convolutional Block (원래 AASIST의 Mel-Spectrogram 처리 부분에 대응)
        # 여기서는 XLS-R 임베딩 (sequence_length, xlsr_model_dimension)을 시계열 데이터로 간주하고 처리
        self.conv_block = nn.Sequential(
            nn.Conv1d(xlsr_model_dimension, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),

            nn.Conv1d(128, 256, kernel_size=5, padding=2),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
        )

        # GRU Block (시계열 특징 학습)
        # Conv Block의 출력 채널과 시퀀스 길이를 고려해야 합니다.
        # MaxPool1d를 두 번 거치면 시퀀스 길이가 4로 나눠집니다. (대략)
        # 예: 249 -> (249+2*1-5)/2 + 1 = 124.5 -> 124 -> (124+2*1-5)/2 + 1 = 61.5 -> 61
        # 실제로는 XLS-R 출력의 sequence_length가 (input_length - 1) // feature_extractor.config.stride + 1 이므로
        # 약 249 (for 80000 samples)
        # 249 -> MaxPool1d(ks=3, s=2, p=1) => floor((249 + 2*1 - 3)/2) + 1 = floor(248/2) + 1 = 124 + 1 = 125
        # 125 -> MaxPool1d(ks=3, s=2, p=1) => floor((125 + 2*1 - 3)/2) + 1 = floor(124/2) + 1 = 62 + 1 = 63

        # 따라서 GRU의 input_size는 conv_block의 마지막 출력 채널 (256)이 됩니다.
        self.gru = nn.GRU(
            input_size=256,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(128 * 2, 128), # Bidirectional GRU이므로 hidden_size * 2
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x_raw_audio, xlsr_model, feature_extractor):
        # 1. XLS-R Feature Extractor를 통해 raw audio를 input_values로 변환
        # (batch_size, sequence_length) -> (batch_size, 1, sequence_length) (if only one channel)

        # raw_audio는 VoiceXLSRDataset에서 (sequence_length)로 나오므로
        # feature_extractor에는 numpy array로, 배치 처리 시에는 리스트로 묶어서 전달

        # x_raw_audio: (batch_size, MAX_AUDIO_LENGTH)
        # inputs = feature_extractor(x_raw_audio.cpu().numpy(), sampling_rate=XLSR_SAMPLE_RATE, return_tensors="pt", padding=True, truncation=True)
        # input_values = inputs.input_values.to(device)

        # 위는 단일 오디오를 위한 코드. 여기서는 배치로 들어온 raw audio를 처리해야 함.
        # batch_size가 4인 경우, x_raw_audio는 (4, MAX_AUDIO_LENGTH)

        # Wav2Vec2FeatureExtractor는 리스트 of numpy array를 받거나, 2D numpy array (batch_size, audio_length)
        # 또는 2D torch tensor를 받음.
        inputs = feature_extractor(x_raw_audio.cpu().numpy(), sampling_rate=XLSR_SAMPLE_RATE, return_tensors="pt", padding=True, truncation=True)
        input_values = inputs.input_values.to(device)

        # 2. XLS-R 모델을 통해 embedding 추출
        # output_hidden_states=True로 설정하여 모든 레이어의 hidden states를 가져올 수 있음
        with torch.no_grad(): # XLS-R 자체는 훈련시키지 않으므로 no_grad
            xlsr_outputs = xlsr_model(input_values, output_hidden_states=True)
            # last_hidden_state: (batch_size, sequence_length, hidden_size)
            xlsr_embedding = xlsr_outputs.last_hidden_state # 또는 특정 레이어의 hidden_state

        # (batch_size, sequence_length, hidden_size) -> (batch_size, hidden_size, sequence_length) for Conv1d
        x = xlsr_embedding.permute(0, 2, 1)

        # Conv Block
        x = self.conv_block(x)

        # (batch_size, channels, sequence_length) -> (batch_size, sequence_length, channels) for GRU
        x = x.permute(0, 2, 1)

        # GRU Block
        output, _ = self.gru(x)

        # 마지막 시점의 출력을 사용
        last_output = output[:, -1, :]

        # Classifier
        logits = self.classifier(last_output)

        return logits

print("AASIST 모델 정의 완료 (XLS-R 임베딩 사용)")

### 셀 46: XLS-R + AASIST 고도화 관련

이 셀은 사전학습 음성 모델을 활용한 고도화 실험 단계입니다. 기본 모델보다 무겁기 때문에 별도 실행을 권장합니다.


In [ ]:
# NOTE: XLS-R + AASIST 모델 학습 루프입니다. GPU 메모리 사용량이 크므로 배치 크기를 작게 유지하세요.
# ============================================================
# 39. XLS-R + AASIST 모델 학습 루프 (상세 주석 버전)
# ============================================================
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm
import torch
import torch.nn as nn

# --- [오류 수정 패치] ---
# 이전 셀에서 정의한 AASISTEmbeddingsClassifier의 forward 함수에서
# truncation=True 사용 시 max_length가 누락되어 발생하는 오류를 수정합니다.
def patched_forward(self, x_raw_audio, xlsr_model, feature_extractor):
    # max_length=MAX_AUDIO_LENGTH 옵션 추가
    inputs = feature_extractor(
        x_raw_audio.cpu().numpy(),
        sampling_rate=XLSR_SAMPLE_RATE,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_AUDIO_LENGTH
    )
    input_values = inputs.input_values.to(device)

    with torch.no_grad():
        xlsr_outputs = xlsr_model(input_values, output_hidden_states=True)
        xlsr_embedding = xlsr_outputs.last_hidden_state

    x = xlsr_embedding.permute(0, 2, 1)
    x = self.conv_block(x)
    x = x.permute(0, 2, 1)
    output, _ = self.gru(x)
    last_output = output[:, -1, :]
    logits = self.classifier(last_output)
    return logits

# 기존 모델 클래스의 forward 메서드를 수정된 함수로 교체합니다.
AASISTEmbeddingsClassifier.forward = patched_forward
# --------------------------

# 1. 모델, 손실 함수, 옵티마이저 초기화
# 설계한 AASIST 임베딩 분류기 모델을 생성하고 GPU(또는 CPU) 메모리로 이동시킵니다.
xlsr_aasist_model = AASISTEmbeddingsClassifier().to(device)

# 손실 함수 설정: 이진/다중 분류에 범용적으로 쓰이는 Cross Entropy Loss를 사용합니다.
criterion_xlsr = nn.CrossEntropyLoss()

# 옵티마이저 설정: Adam 최적화 알고리즘을 사용합니다.
# 학습률(Learning rate)은 모델이 섬세하게 가중치를 업데이트하도록 1e-4(0.0001)로 설정합니다.
optimizer_xlsr = torch.optim.Adam(xlsr_aasist_model.parameters(), lr=1e-4)

# 학습 설정값
EPOCHS = 5  # 전체 학습 데이터셋을 몇 번 반복해서 학습할지 설정합니다.
best_val_f1_xlsr = 0  # 가장 성능이 좋은 시점을 찾기 위해 최고 F1 점수를 기록할 변수입니다.
save_dir_xlsr = "/content/drive/MyDrive/딥러닝-석흥일/프로젝트_보이스피싱탐지"
best_model_path_xlsr = f"{save_dir_xlsr}/XLSR_AASIST_best.pt"  # 모델 가중치를 저장할 구글 드라이브 경로

xlsr_history = []  # 매 에폭(Epoch)마다 변화하는 손실 및 성능 지표를 저장할 리스트입니다.

print("🚀 XLS-R + AASIST 모델 학습을 시작합니다...")

# 에폭(Epoch) 단위 반복 시작
for epoch in range(1, EPOCHS + 1):
    print(f"\n===== XLS-R + AASIST Epoch {epoch}/{EPOCHS} =====")

    # ------------------ Train (학습 단계) ------------------
    xlsr_aasist_model.train()  # 모델을 학습 모드로 변경합니다. (Dropout 등이 활성화됨)
    total_train_loss = 0  # 현재 에폭의 총 손실값을 누적할 변수
    train_preds, train_labels = [], []  # 예측값과 실제 정답을 모아둘 리스트

    # 학습용 데이터로더에서 배치(Batch) 단위로 데이터를 꺼내옵니다.
    for x, y in tqdm(train_loader_xlsr, desc="Train"):
        x, y = x.to(device), y.to(device)  # 데이터(x)와 라벨(y)을 GPU로 이동

        optimizer_xlsr.zero_grad()  # 이전 배치에서 계산된 기울기(Gradient)가 누적되지 않도록 초기화

        # 순전파(Forward Pass): 모델에 데이터, XLS-R 추출기, 피처 추출기를 함께 전달해 결과를 예측
        outputs = xlsr_aasist_model(x, xlsr_model, feature_extractor)

        # 오차(Loss) 계산: 모델의 예측값(outputs)과 실제 정답(y)의 차이를 계산
        loss = criterion_xlsr(outputs, y)

        loss.backward()  # 역전파(Backward Pass): 오차를 바탕으로 각 가중치의 기울기를 계산
        optimizer_xlsr.step()  # 옵티마이저가 계산된 기울기를 사용해 모델의 가중치를 업데이트

        total_train_loss += loss.item()  # 로스값을 텐서에서 숫자(스칼라)로 뽑아내어 누적

        # 모델 출력 중 확률이 가장 높은 클래스의 인덱스를 찾아 최종 예측값으로 결정
        preds = torch.argmax(outputs, dim=1)

        # CPU 메모리로 옮긴 후 리스트에 추가 (평가지표 계산용)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels.extend(y.detach().cpu().numpy())

    # 에폭 평균 학습 Loss 및 정확도 계산
    train_loss = total_train_loss / len(train_loader_xlsr)
    train_acc = accuracy_score(train_labels, train_preds)

    # ------------------ Validation (검증 단계) ------------------
    xlsr_aasist_model.eval()  # 모델을 평가 모드로 변경합니다. (Dropout 비활성화, 고정된 환경에서 평가)
    total_val_loss = 0
    val_preds, val_labels = [], []

    # 평가 단계에서는 가중치를 업데이트하지 않으므로 기울기 계산을 끕니다. (메모리 절약, 속도 향상)
    with torch.no_grad():
        for x, y in tqdm(val_loader_xlsr, desc="Val"):
            x, y = x.to(device), y.to(device)

            # 검증 데이터에 대한 예측 및 손실 계산
            outputs = xlsr_aasist_model(x, xlsr_model, feature_extractor)
            loss = criterion_xlsr(outputs, y)

            total_val_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)

            val_preds.extend(preds.detach().cpu().numpy())
            val_labels.extend(y.detach().cpu().numpy())

    # 에폭 평균 검증 손실 및 여러 평가 지표 계산
    val_loss = total_val_loss / len(val_loader_xlsr)
    val_acc = accuracy_score(val_labels, val_preds)
    val_precision = precision_score(val_labels, val_preds, zero_division=0)  # 정밀도
    val_recall = recall_score(val_labels, val_preds, zero_division=0)  # 재현율
    val_f1 = f1_score(val_labels, val_preds, zero_division=0)  # F1 Score (정밀도와 재현율의 조화평균)

    # 화면에 결과 출력
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val Precision: {val_precision:.4f} | Val Recall: {val_recall:.4f} | Val F1: {val_f1:.4f}")

    # 분석을 위해 히스토리 리스트에 딕셔너리 형태로 결과 저장
    xlsr_history.append({
        "epoch": epoch,
        "train_loss": train_loss, "train_acc": train_acc,
        "val_loss": val_loss, "val_acc": val_acc,
        "val_precision": val_precision, "val_recall": val_recall, "val_f1": val_f1
    })

    # Best 모델 저장 로직
    # 현재 에폭의 검증 F1 Score가 이전 최고 점수보다 높다면 기록을 갱신하고 모델 저장
    if val_f1 > best_val_f1_xlsr:
        best_val_f1_xlsr = val_f1
        torch.save(xlsr_aasist_model.state_dict(), best_model_path_xlsr)
        print("⭐ Best model saved:", best_model_path_xlsr)

print("\n✅ 학습이 완료되었습니다!")


### 셀 39: XLS-R + AASIST 고도화 관련

이 셀은 사전학습 음성 모델을 활용한 고도화 실험 단계입니다. 기본 모델보다 무겁기 때문에 별도 실행을 권장합니다.


In [ ]:
# NOTE: XLS-R 어텐션이 음성의 어느 구간에 집중하는지 시각화합니다.
import torchaudio
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# 오디오 샘플 로드
# 이전 단계에서 사용한 실제 음성 샘플 경로를 재활용합니다.
# real_path: /content/data_kspon/wav_real/train/KsponSpeech_0007/KsponSpeech_006050.wav
audio_sample, sample_rate = torchaudio.load(real_path)

# XLS-R 모델은 16kHz 샘플레이트를 기대합니다.
if sample_rate != XLSR_SAMPLE_RATE:
    resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=XLSR_SAMPLE_RATE)
    audio_sample = resampler(audio_sample)

# 정해진 최대 길이(80000)로 오디오를 자르거나 패딩합니다.
if audio_sample.shape[1] > MAX_AUDIO_LENGTH:
    audio_sample = audio_sample[:, :MAX_AUDIO_LENGTH]
else:
    padding = MAX_AUDIO_LENGTH - audio_sample.shape[1]
    audio_sample = torch.nn.functional.pad(audio_sample, (0, padding))

# XLS-R Feature Extractor를 사용하여 입력 피처를 준비합니다.
# squeeze(0)는 단일 채널 오디오의 배치 차원을 제거합니다.
inputs = feature_extractor(audio_sample.squeeze(0).numpy(), sampling_rate=XLSR_SAMPLE_RATE, return_tensors="pt")
input_values = inputs.input_values.to(device)

# 모델을 통해 어텐션 가중치를 얻습니다.
# output_attentions=True가 모델 로드 시 설정되어 있어야 합니다.
with torch.no_grad():
    outputs = xlsr_model(input_values, output_attentions=True)
    attentions = outputs.attentions  # 각 레이어의 어텐션 튜플

# 마지막 레이어의 어텐션 가중치를 가져와 헤드별로 평균을 냅니다.
# attentions[-1]는 마지막 레이어의 어텐션입니다: (batch_size, num_heads, sequence_length, sequence_length)
last_layer_attention = attentions[-1].squeeze(0)  # 배치 차원 제거
avg_attention = last_layer_attention.mean(dim=0).cpu().numpy() # 헤드별 평균 및 CPU로 이동

print("어텐션 맵 shape:", avg_attention.shape)

# 어텐션 맵 시각화
plt.figure(figsize=(12, 10))
sns.heatmap(avg_attention, cmap='viridis')
plt.title('XLS-R 마지막 레이어 평균 어텐션 맵')
plt.xlabel('키 포지션')
plt.ylabel('쿼리 포지션')
plt.show()

### 💡 어텐션 비쥬얼라이제이션 (Attention Visualization) 상세 설명

이 과정은 사전 학습된 XLS-R 모델이 음성 데이터를 처리할 때 **어느 시간대(Time step)의 특징에 집중(Attention)하는지**를 시각적으로 보여줍니다.

**1. 오디오 데이터 전처리**
- 실제 음성 샘플을 로드한 뒤, XLS-R 모델이 요구하는 **16kHz 샘플레이트**로 변환합니다.
- 모델 입력 길이를 맞추기 위해 최대 5초(80,000 샘플) 기준으로 자르거나(Truncation) 모자란 부분은 0으로 채웁니다(Padding).

**2. 어텐션 가중치(Attention Weights) 추출**
- Feature Extractor를 거친 데이터를 모델에 통과시킵니다. 이때 `output_attentions=True` 설정을 통해 모델 내부의 어텐션 연산 결과를 반환받습니다.

**3. 마지막 레이어 및 헤드 평균 산출**
- 트랜스포머(Transformer) 구조는 여러 개의 레이어와 헤드(Multi-Head)를 가집니다. 여기서는 모델의 최종 결정에 가장 큰 영향을 미치는 **마지막 레이어**의 어텐션 가중치를 가져온 뒤, 모든 헤드의 값을 평균 내어 하나의 2D 맵으로 만듭니다.

**4. 히트맵(Heatmap) 해석**
- **X축 (키 포지션)** 및 **Y축 (쿼리 포지션)**: 오디오 시퀀스 상의 시간적 위치를 나타냅니다.
- **밝은 색상 (노란색/연두색)**: 가중치가 높은 부분으로, 모델이 해당 시점의 음성 특징 간의 상관관계를 매우 중요하게 보고 **강하게 집중(Attend)** 하고 있음을 뜻합니다.
- **어두운 색상 (보라색/남색)**: 가중치가 낮은 부분으로, 비교적 정보의 중요도나 연관성이 낮다고 판단한 구간입니다.

## 10. 참고 자료

아래 자료는 PyTorch attention 함수 등 추가 학습에 참고할 수 있는 링크입니다.


## 6. Resources
### 6-1. 어텐션 비쥬얼라이제이션 (Attention Visualization)

### 6-2. 자료 더보기 (More Resources)
*   [PyTorch Scaled Dot Product Attention (SDPA) Documentation](https://pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)


In [ ]:
!pip freeze > "/content/drive/MyDrive/딥러닝-석흥일/프로젝트_보이스피싱탐지/requirements.txt"